# Joan and Karin: from observed uncertainty to robust cones

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/joan-karin-robust-optimization.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/joan-karin-robust-optimization.ipynb)

By Joaquim Gromicho. Modernized from the original teaching notebook.

Follow the original trophy case through nominal planning, observed wood use, robust box and budgeted models, and continuous/mixed-integer second-order cones. The trophy producer is called Caroline consistently with the ABC sequence.


# Accompanying notebook to Conic and Robust optimization

In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'pyomo': 'pyomo', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
ensure_packages(required_packages)


In [ ]:
# Open-source engines enter first. Commercial conic solvers are introduced later.
from teaching_utils import install_coin_solvers,make_solver,solve_checked,available_pyomo_solvers
install_coin_solvers()


In [ ]:
print(available_pyomo_solvers(['cbc','ipopt']))


In [ ]:
import pyomo.environ as pyo


In [ ]:
%matplotlib inline

# Recall the Caroline's production planning model

Caroline owns a company that produces trophies for
* football
 * wood base, engraved plaque, brass football on top
 * €12 profit and uses 4 dm of wood
* golf
 * wood base, engraved plaque, golf ball on top
 * €9 profit and uses 2 dm of wood

Caroline’s current stock of raw materials
* 1000 footballs
* 1500 golf balls
* 1750 plaques
* 480 m (4800 dm) of wood

> Caroline wonders what the optimal production plan should be, in other words: how many football and how many golf trophies should Caroline produce to maximize his profit while respecting the availability of raw materials?

***

The following model __maximizes__ Caroline's profit by deciding the number of $x_1$ football and $x_2$ golf trophies to produce.

$$
\begin{array}{rrcrcl}
\max    & 12x_1 & + & 9x_2               \\
s.t.    &   x_1 &   &      & \leq & 1000 \\
        &       &   &  x_2 & \leq & 1500 \\
        &   x_1 & + &  x_2 & \leq & 1750 \\
        &  4x_1 & + & 2x_2 & \leq & 4800 \\
        &   x_1 & , &  x_2 & \geq & 0    \\
\end{array}
$$

In [ ]:
trophies = [ 'Football', 'Golf' ]
profits  = { 'Football' : 12, 'Golf' :  9 }
wood     = { 'Football' :  4, 'Golf' :  2 }

Caroline = pyo.ConcreteModel('Caroline')

Caroline.x = pyo.Var(trophies,within=pyo.NonNegativeReals)

Caroline.profit    = pyo.Objective(expr = sum([profits[t]*Caroline.x[t] for t in trophies]), sense=pyo.maximize)

Caroline.footballs = pyo.Constraint(expr = Caroline.x['Football']  <= 1000)
Caroline.golfBalls = pyo.Constraint(expr = Caroline.x['Golf']      <= 1500)
Caroline.plaques   = pyo.Constraint(expr = sum([Caroline.x[t] for t in trophies]) <= 1750)
Caroline.wood      = pyo.Constraint(expr = sum(wood[t]*Caroline.x[t] for t in trophies) <= 4800 )

In [ ]:
%time results = solve_checked(Caroline,'cbc')
print(results.solver.status, results.solver.termination_condition )

print(Caroline.profit.expr())
print([Caroline.x[t].value for t in trophies])

Caroline.display()

In [ ]:
Caroline.pprint()

In [ ]:
# Standard Pyomo inspection suffices here; Caroline's LP lesson explains components.
# Reuse the modeling library's built-in reporting instead of copying its traversal code.


In [ ]:
Caroline.display()


In [ ]:
def ShowDuals(model):
    import pandas as pd
    return pd.Series({constraint.name:float(model.dual[constraint])
                      for constraint in model.component_data_objects(pyo.Constraint,active=True)},name='dual')


In [ ]:
Caroline.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

%time results = solve_checked(Caroline,'cbc')
print(results.solver.status, results.solver.termination_condition )

In [ ]:
ShowDuals( Caroline )

In [ ]:
def JustSolution( model ):
  return [ pyo.value(model.profit) ] + [ pyo.value(model.x[i]) for i in trophies ]

In [ ]:
JustSolution( Caroline )

## Joan: a tiny bit of data science...


We start by simulating two samples of observed wood lengths for `f` football trophies and `g` golf trophies.

In [ ]:
import numpy as np
rng=np.random.default_rng(2021)
n=2000
f=rng.lognormal(np.log(4.),.005,n)
g=rng.lognormal(np.log(2.),.005,n)


In [ ]:
print('First observations:',f[:10],g[:10])


In [ ]:
print( min(f), max(f), min(g), max(g) )

In [ ]:
import matplotlib.pyplot as plt

plt.plot( f, '.' )
plt.plot( g, '.' )
plt.show()

What is the consequence of uncertainty?
We compare the cumulative lengths with the nominal ones.

In [ ]:
cs = np.cumsum(g) - np.cumsum( [2]*len(g) )
plt.plot(cs)
plt.show()

In [ ]:
plt.pie( [ sum( cs > 0 ), sum( cs <= 0 ) ], labels = [ 'trouble!', 'ok' ], autopct='%1.1f%%', shadow=True, startangle=90, colors=[ 'red', 'green' ])
plt.show()

A very simple and somehow naïf uncertainty region can be taken as the observed minimal box around the data.
An observed bounding box describes this sample. It does not guarantee coverage of future observations from an unbounded distribution. Robust feasibility below is conditional on the chosen uncertainty set.


In [ ]:
import matplotlib.patches as patches

plt.figure()
plt.plot( f, g, '.' )
currentAxis = plt.gca()
currentAxis.add_patch(patches.Rectangle((min(f), min(g)), max(f)-min(f), max(g)-min(g),fill=False,color='r'))
plt.show()

print( min(f), max(f), min(g), max(g) )

## Karin's robust model for box uncertainty in wood consumption

Suppose now that Caroline notices that not _exactly_ 4 and 2 dm of wood are used, but some fluctuations are observed.
Caroline wants to be __sure__ that her model does not violate the wood constraint, therefore the following should hold:

$$
\begin{array}{rrcrcl}
\max    &  12 x_1 & + &   9 x_2               \\
s.t.    &     x_1 &   &         & \leq & 1000 \\
        &         &   &     x_2 & \leq & 1500 \\
        &     x_1 & + &     x_2 & \leq & 1750 \\
        & a_1 x_1 & + & a_2 x_2 & \leq & 4800 & \forall \ell \leq a \leq u \\
        &     x_1 & , &     x_2 & \geq & 0    \\
\end{array}
$$

***

A bit of linear duality (or even better: an introduction to robust optimization!) helps Caroline how to deal with the above model that has an infinite number of constraints.
The first thing to notice is that the wood consumption is modeled by constraints that are equivalent to bounding the following optimization problem:
    
$$
\begin{array}{rrr}
\max    & x_1 a_1 + x_2 a_2 & \leq 4800 \\
s.t.    & \ell \leq a \leq u
\end{array}
$$

Or

$$
\begin{array}{rrr}
\max    & x_1 a_1 + x_2 a_2 & \leq 4800 \\
s.t.    & a \leq u \\
        & -a \leq -\ell
\end{array}
$$

Now we use linear duality to realize that the above is equivalent to:

$$
\begin{array}{rrr}
\min    & u y  - \ell w & \leq 4800 \\
s.t.    & y - w = x \\
        & y \geq 0, w \geq 0
\end{array}
$$
    
and the constraint imposed by the last problem is equivalent to:

$$
\begin{array}{rrl}
   & u y  - \ell w & \leq 4800 \\
   & y - w & = x \\
   & y \geq 0, w \geq 0
\end{array}
$$

The only thing we need to do is add variables and constraints to Caroline's model.

# A model in `pyomo`

In [ ]:
def CarolineWithBoxUncertainty( lower, upper, domain=pyo.NonNegativeReals ):

    Caroline = pyo.ConcreteModel('CarolineBox')

    Caroline.x = pyo.Var(trophies,within=domain)

    Caroline.profit    = pyo.Objective(expr = sum([profits[t]*Caroline.x[t] for t in trophies]), sense=pyo.maximize)

    Caroline.footballs = pyo.Constraint(expr = Caroline.x['Football']  <= 1000)
    Caroline.golfBalls = pyo.Constraint(expr = Caroline.x['Golf']      <= 1500)
    Caroline.plaques   = pyo.Constraint(expr = sum([Caroline.x[t] for t in trophies]) <= 1750)

    Caroline.y = pyo.Var(trophies,domain=pyo.NonNegativeReals)
    Caroline.w = pyo.Var(trophies,domain=pyo.NonNegativeReals)

    Caroline.robustWood = pyo.Constraint(expr = sum([upper[t]*Caroline.y[t] - lower[t]*Caroline.w[t] for t in trophies]) <= 4800)

    def PerVariable( model, t ):
        return model.x[t] == model.y[t] - model.w[t]

    Caroline.perVariable = pyo.Constraint(trophies,rule=PerVariable)

    return Caroline

In [ ]:
lower = {}
upper = {}
lower['Football'] = min(f)
upper['Football'] = max(f)
lower['Golf'] = min(g)
upper['Golf'] = max(g)

Caroline = CarolineWithBoxUncertainty( lower, upper, domain=pyo.NonNegativeReals )

%time results = solve_checked(Caroline,'cbc')
print(results.solver.status, results.solver.termination_condition )

JustSolution( Caroline )

In [ ]:
# you can play with the amount of uncertainty.
# In particular, if below you make delta equal to 0 you obtain the same result als the  nominal model.
delta = 0.05

def CarolineWithSymmetricalBoxUncertainty( delta, domain=pyo.NonNegativeIntegers ):
    lower = { trophy : wood[trophy] - delta for trophy in wood }
    upper = { trophy : wood[trophy] + delta for trophy in wood }
    return CarolineWithBoxUncertainty( lower, upper, domain=domain )

Caroline = CarolineWithSymmetricalBoxUncertainty( delta )
%time results = solve_checked(Caroline,'cbc')
print(results.solver.status, results.solver.termination_condition )

JustSolution( Caroline )

# Integer optimization

The nominal LP happens to have an integer optimum; a robust variant need not. Declare integer production explicitly when trophies cannot be fractional. A box counterpart stays linear, so CBC can solve the resulting MILP.


In [ ]:
Caroline = CarolineWithBoxUncertainty( lower, upper, domain=pyo.NonNegativeIntegers )

%time results = solve_checked(Caroline,'cbc')
print(results.solver.status, results.solver.termination_condition )

JustSolution( Caroline )

In [ ]:
import pandas
df = pandas.DataFrame()
for delta in np.linspace(0,.5,21):
  Caroline = CarolineWithSymmetricalBoxUncertainty( delta, domain=pyo.NonNegativeIntegers )
  solve_checked(Caroline,'cbc')
  results = JustSolution( Caroline )
  df.at[delta,'value']     = results[0]
  df.at[delta,trophies[0]] = results[1]
  df.at[delta,trophies[1]] = results[2]
df

In [ ]:
df.plot()

In [ ]:
df[['Football','Golf']].plot()

# Cardinality constrained uncertainty

Each $a_j$ may deviate by at most $\pm \delta_j$ from the nominal value $\bar{a}_j$ with a total normalized deviation budget $\Gamma$. For noninteger $\Gamma$, this permits fractional deviations rather than literally counting changed coefficients.

$$
\begin{array}{rrcrcl}
\max    &  12 x_1 & + &   9 x_2               \\
s.t.    &     x_1 &   &         & \leq & 1000 \\
        &         &   &     x_2 & \leq & 1500 \\
        &     x_1 & + &     x_2 & \leq & 1750 \\
        & a_1 x_1 & + & a_2 x_2 & \leq & 4800 & \forall a,y : a_j=\bar{a}_j+\delta_jy_j, \|y\|_\infty \leq 1, \|y\|_1\leq \Gamma \\
        &     x_1 & , &     x_2 & \geq & 0    \\
\end{array}
$$

As we have seen on the previous lecture, Lagrange duality yields the following modification to the problem as equivalent to the robust model stated above:

$$
\begin{array}{rrcrcrcrcrcrcl}
\max    &  12 x_1 & + &   9 x_2               \\
s.t.    &     x_1 &   &         & & & & & & & \leq & 1000 \\
        &         &   &     x_2 & & & & & & & \leq & 1500 \\
        &     x_1 & + &     x_2 & & & & & & & \leq & 1750 \\
        & a_1 x_1 & + & a_2 x_2 & + & \lambda\Gamma & + & z_1 & + & z_2 & \leq & 4800 \\
        &-d_1 x_1 &   &         & + & \lambda & + & z_1 &   &     & \geq & 0 \\
        &         &   &-d_2 x_2 & + & \lambda &   &     & + & z_2 & \geq & 0 \\
        &     x_1 & , &     x_2 & , & \lambda & , & z_1 & , & z_2 & \geq & 0    \\
\end{array}
$$

In [ ]:
def CarolineWithGammaUncertainty( delta, gamma, domain=pyo.NonNegativeReals ):
    Caroline = pyo.ConcreteModel('CarolineGamma')

    Caroline.x = pyo.Var(trophies,within=domain)

    Caroline.profit    = pyo.Objective(expr = sum([profits[t]*Caroline.x[t] for t in trophies]), sense=pyo.maximize)

    Caroline.footballs = pyo.Constraint(expr = Caroline.x['Football']  <= 1000)
    Caroline.golfBalls = pyo.Constraint(expr = Caroline.x['Golf']      <= 1500)
    Caroline.plaques   = pyo.Constraint(expr = sum([Caroline.x[t] for t in trophies]) <= 1750)

    Caroline.z   = pyo.Var(trophies,domain=pyo.NonNegativeReals)
    Caroline.lam = pyo.Var(domain=pyo.NonNegativeReals)

    Caroline.robustWood = pyo.Constraint( \
     expr = sum([wood[t]*Caroline.x[t] for t in trophies]) \
          + gamma * Caroline.lam \
          + sum(Caroline.z[t] for t in trophies) <= 4800)

    def up_rule( model, t ):
        return model.z[t] >=  delta * model.x[t] - model.lam
    def down_rule( model, t ):
        return model.z[t] >= -delta * model.x[t] - model.lam

    Caroline.up   = pyo.Constraint(trophies,rule=up_rule)
    Caroline.down = pyo.Constraint(trophies,rule=down_rule)

    return Caroline

In [ ]:
Caroline = CarolineWithGammaUncertainty( .05, 2, domain=pyo.NonNegativeIntegers )

%time results = solve_checked(Caroline,'cbc')
print(results.solver.status, results.solver.termination_condition )
JustSolution(Caroline)

# Ball uncertainty

As [the documentation](https://pyomo.readthedocs.io/en/stable/reference/topical/kernel/conic.html) says a conic constraint is expressed in 'pyomo' in simple variables.

This [table](https://pyomo.readthedocs.io/en/stable/reference/topical/kernel/index.html) is very useful.

A straightforward remodulation leads to that:

$$
  a_1x_1+a_2x_2 + \Omega \|x\| \leq 4800
$$

$$
  \Omega \|x\| \leq 4800 - a_1x_1 - a_2x_2
$$
$$
  \|\Omega x\| \leq 4800 - a_1x_1 - a_2x_2
$$

By defining $y = 4800 - a_1x_1 - a_2x_2$ we may write:
$$
  \Omega^2 \|x\|^2 \leq y^2
$$

$$
  (\Omega x_1)^2 + (\Omega x_2)^2 \leq y^2
$$

$$
  \|w\|^2 \leq y^2
$$

with $w = \Omega x$.
The squared inequality is equivalent only with y>=0. Both formulations below enforce that sign condition explicitly.


In [ ]:
# we need to use the kernel now...
import pyomo.kernel as pyk

def CarolineWithBallUncertainty( omega, domain_type=pyk.RealSet ):

    idxTrophies = range( len(trophies) )

    Caroline = pyk.block()

    Caroline.x = pyk.variable_list()
    for i in idxTrophies:
        Caroline.x.append( pyk.variable(lb=0,domain_type=domain_type) )

    Caroline.profit    = pyk.objective(expr = sum(profits[trophies[i]]*Caroline.x[i] for i in idxTrophies), sense=pyk.maximize)

    Caroline.footballs = pyk.constraint(expr = Caroline.x[0]  <= 1000)
    Caroline.golfBalls = pyk.constraint(expr = Caroline.x[1]  <= 1500)
    Caroline.plaques   = pyk.constraint(expr = sum([Caroline.x[i] for i in idxTrophies]) <= 1750)

    Caroline.y = pyk.variable(lb=0)
    Caroline.w = pyk.variable_list()
    for i in idxTrophies:
        Caroline.w.append( pyk.variable(lb=0) )

    Caroline.wood = pyk.constraint( expr = Caroline.y == 4800 - sum(wood[trophies[i]]*Caroline.x[i] for i in idxTrophies) )

    Caroline.xtow = pyk.constraint_list()
    for i in idxTrophies:
        Caroline.xtow.append( pyk.constraint( expr = Caroline.w[i] == omega * Caroline.x[i] ) )

    from pyomo.core.kernel.conic import quadratic
    Caroline.robust = quadratic(Caroline.y,Caroline.w)

    return Caroline

## Now the problem is nonlinear

In [ ]:
from time import perf_counter
Caroline=CarolineWithBallUncertainty(0.1)
started=perf_counter()
results=solve_checked(Caroline,'ipopt')
ipopt_ball_seconds=perf_counter()-started
ipopt_ball_value=pyk.value(Caroline.profit)
print(results.solver.termination_condition,ipopt_ball_value,[pyk.value(x) for x in Caroline.x])


## Introduce CPLEX, Gurobi and Xpress for conic optimization

We have solved the continuous cone model with Ipopt. Now install the commercial interfaces and actually run all three engines on fresh continuous and integer models. The tiny instances fit the packaged limited editions. If an engine cannot solve, stop and resolve the error; do not report a skipped solver as a comparison result.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'cplex': 'cplex', 'gurobipy': 'gurobipy', 'xpress': 'xpress'}
ensure_packages(required_packages)


In [ ]:
conic_engines=('gurobi_direct','cplex_direct','xpress_direct')
conic_comparison=[dict(engine='ipopt',form='kernel',integer=False,objective=ipopt_ball_value,seconds=ipopt_ball_seconds)]


In [ ]:
from time import perf_counter
for engine in conic_engines:
    model=CarolineWithBallUncertainty(0.1)
    started=perf_counter()
    solve_checked(model,engine)
    objective=pyk.value(model.profit)
    conic_comparison.append(dict(engine=engine,form='kernel',integer=False,objective=objective,seconds=perf_counter()-started))
    print(engine,objective,[pyk.value(x) for x in model.x])


## And therefore we can also have mixed integer models

In [ ]:
for engine in conic_engines:
    model=CarolineWithBallUncertainty(0.1,domain_type=pyk.IntegerSet)
    started=perf_counter()
    solve_checked(model,engine)
    objective=pyk.value(model.profit)
    conic_comparison.append(dict(engine=engine,form='kernel',integer=True,objective=objective,seconds=perf_counter()-started))
    print(engine,objective,[pyk.value(x) for x in model.x])


## Final note: maybe useful to recall that in python you can always ask for help...

In [ ]:
print('The cone-side variable has lower bound:',model.y.lb)


In [ ]:
print('Conic constraint:',model.robust)


# How to bring second order cones into the `pyomo.environ`

Noting that $\| x \| \leq t$ is for $t \geq 0$ equivalent to $\| x \|^2 \leq t^2$ and knowing that the commercial solvers (`gurobi`, `cplex` and `express`) support convex quadratic inequalities, we can model this variant in `pyomo.environ` as follows.

Note that the essential part to make the model convex is having the right hand side nonnegative.

In [ ]:
def CarolineWithBallUncertaintyAsSquaredSecondOrderCone(omega,domain=pyo.NonNegativeReals):
  Caroline = pyo.ConcreteModel('Caroline')

  Caroline.x = pyo.Var(trophies,within=domain)

  # the nonegativity of this variable is essential!
  Caroline.y = pyo.Var(within=pyo.NonNegativeReals)

  Caroline.profit    = pyo.Objective(expr = sum([profits[t]*Caroline.x[t] for t in trophies]), sense=pyo.maximize)

  Caroline.footballs = pyo.Constraint(expr = Caroline.x['Football']  <= 1000)
  Caroline.golfBalls = pyo.Constraint(expr = Caroline.x['Golf']      <= 1500)
  Caroline.plaques   = pyo.Constraint(expr = sum([Caroline.x[t] for t in trophies]) <= 1750)
  Caroline.wood      = pyo.Constraint(expr = Caroline.y == 4800 - sum(wood[t]*Caroline.x[t] for t in trophies) )
  Caroline.robust    = pyo.Constraint(expr = sum((omega*Caroline.x[t])**2 for t in trophies) <= Caroline.y**2)
  return Caroline

In [ ]:
for engine in conic_engines:
    for integer in (False,True):
        domain=pyo.NonNegativeIntegers if integer else pyo.NonNegativeReals
        model=CarolineWithBallUncertaintyAsSquaredSecondOrderCone(0.1,domain=domain)
        started=perf_counter()
        solve_checked(model,engine)
        conic_comparison.append(dict(engine=engine,form='squared',integer=integer,objective=pyo.value(model.profit),seconds=perf_counter()-started))
        x=np.array([pyo.value(model.x[t]) for t in trophies])
        assert np.dot([4,2],x)+0.1*np.linalg.norm(x)<=4800+1e-3
        assert pyo.value(model.y)>=-1e-7


The two cone representations should agree. Check that each engine actually solved each model; compare the integer optimum with independent enumeration on the original two-product bounds.


In [ ]:
comparison=pandas.DataFrame(conic_comparison)
display(comparison)
assert len(comparison)==13 and comparison.engine.nunique()==4
for integer in (False,True):
    values=comparison.loc[comparison.integer==integer,'objective']
    assert values.max()-values.min()<0.05
# Independent feasible-grid reference for the mixed-integer conic optimum.
football=np.arange(1001)[:,None]
golf=np.arange(1501)[None,:]
feasible=(football+golf<=1750)&(4*football+2*golf+0.1*np.sqrt(football**2+golf**2)<=4800+1e-9)
reference=np.where(feasible,12*football+9*golf,-np.inf).max()
assert np.allclose(comparison.loc[comparison.integer,'objective'],reference,atol=1e-4)
print('Independent integer conic optimum:',reference)


In [ ]:
# Zero budget recovers nominal wood; Gamma=2 recovers the two-coordinate box.
checks=[]
for gamma in [0,1,2]:
    model=CarolineWithGammaUncertainty(.05,gamma,domain=pyo.NonNegativeIntegers)
    solve_checked(model,'cbc');checks.append(pyo.value(model.profit))
box=CarolineWithSymmetricalBoxUncertainty(.05)
solve_checked(box,'cbc')
assert checks[0]>=checks[1]>=checks[2]
assert abs(checks[2]-pyo.value(box.profit))<1e-5
assert lower is not upper and all(lower[t]<upper[t] for t in trophies)
print('Gamma 0/1/2 objectives:',checks)
